In [1]:
import os
import pickle
import pandas as pd
import nglview as nv
from IPython.display import display

# =========================
# Paths
# =========================
pdb_dir = "/WAVE/bio/ML/SAE_train/SAEProteinMPNN/evaluation/inputs/"

encodings_path = "/WAVE/bio/ML/SAE_train/SAEProteinMPNN/evaluation/created_data/encodings/output_node_log18_2.pkl"

# =========================
# Load encodings
# =========================
dfs = []

with open(encodings_path, "rb") as f:
    while True:
        try:
            dfs.append(pickle.load(f))
        except EOFError:
            break

encodings = pd.concat(dfs, ignore_index=True)

# =========================
# Clean and parse identifiers
# Example identifier: 1a7wA123
# pdb_id = 1a7w
# chain = A
# residue_idx = 123
# =========================
encodings = encodings.dropna(subset=["identifier"]).copy()
encodings["identifier"] = encodings["identifier"].astype(str)

encodings["pdb_id"] = encodings["identifier"].str[:4]
encodings["chain"] = encodings["identifier"].str[4]

encodings["residue_idx"] = pd.to_numeric(
    encodings["identifier"].str[5:],
    errors="coerce"
)

encodings = encodings.dropna(subset=["residue_idx"]).copy()
encodings["residue_idx"] = encodings["residue_idx"].astype(int)

# =========================
# Get dimension columns
# Your dimensions are numeric columns: 1, 2, 3, ..., 256
# =========================
dimension_cols = [col for col in encodings.columns if isinstance(col, int)]

print("Number of dimensions:", len(dimension_cols))
print("First 10 dimensions:", dimension_cols[:10])

# =========================
# Pick the 5 most active dimensions globally
# =========================
top_5_dims = (
    encodings[dimension_cols]
    .max()
    .sort_values(ascending=False)
    .head(5)
    .index
    .tolist()
)

print("Top 5 dimensions:", top_5_dims)

# =========================
# Visualization function
# =========================
def visualize_top_residues_for_dimension(top10, dim_col):
    for pdb_id in top10["pdb_id"].unique():
        subset = top10[top10["pdb_id"] == pdb_id]

        pdb_path = os.path.join(pdb_dir, f"{pdb_id}.pdb")

        if not os.path.exists(pdb_path):
            print(f"Skipping {pdb_id}: PDB file not found")
            continue

        print(f"\nProtein: {pdb_id}")
        print(f"Dimension: {dim_col}")

        display(subset[["identifier", "pdb_id", "chain", "residue_idx", dim_col]])

        view = nv.show_file(pdb_path, ext="pdb")
        view.clear_representations()

        view.add_cartoon(selection="protein", color="lightgray")

        selections = []

        for _, row in subset.iterrows():
            chain = row["chain"]
            residue = int(row["residue_idx"])
            selections.append(f"resi {residue} and :{chain}")

        selection = " or ".join(selections)

        print("Highlighting:", selection)

        view.add_ball_and_stick(selection=selection, color="red")
        view.add_spacefill(selection=selection, color="red", radiusScale=0.6)

        view.center()

        display(view)

# =========================
# Main loop:
# For each of 5 dimensions,
# find top 10 residues across ALL proteins,
# then visualize those proteins.
# =========================
for dim_col in top_5_dims:
    print("\n" + "=" * 80)
    print(f"TOP 10 RESIDUES FOR DIMENSION {dim_col}")
    print("=" * 80)

    top10 = encodings.nlargest(10, dim_col)

    display(top10[["identifier", "pdb_id", "chain", "residue_idx", dim_col]])

    pdb_path = os.path.join(pdb_dir, "1a7w.pdb")


    visualize_top_residues_for_dimension(top10, dim_col)

Number of dimensions: 256
First 10 dimensions: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Top 5 dimensions: [104, 147, 55, 198, 152]

TOP 10 RESIDUES FOR DIMENSION 104


,identifier,pdb_id,chain,residue_idx,104
378380,7djmA605,7djm,A,605,3.90918
420791,8s6nD75,8s6n,D,75,3.83184
420926,8s6nG75,8s6n,G,75,3.82950
420548,8s6nA75,8s6n,A,75,3.79776
30553,1oi0A91,1oi0,A,91,3.78139
7892,1dqnA183,1dqn,A,183,3.76733
8122,1dqnB183,1dqn,B,183,3.75763
420665,8s6nB75,8s6n,B,75,3.75554
32648,1po5A50,1po5,A,50,3.75515
101345,2qbxA190,2qbx,A,190,3.75420



Protein: 7djm
Dimension: 104


,identifier,pdb_id,chain,residue_idx,104
378380,7djmA605,7djm,A,605,3.90918


Highlighting: resi 605 and :A


...

NGLWidget()


Protein: 6wlf
Dimension: 152


,identifier,pdb_id,chain,residue_idx,152
368634,6wlfB57,6wlf,B,57,2.5002


Highlighting: resi 57 and :B


NGLWidget()

In [2]:
print(pdb_path)
print(os.path.exists(pdb_path))
print(os.path.getsize(pdb_path))

view = nv.show_file(pdb_path, ext="pdb")
view.clear_representations()
view.add_cartoon(selection="protein", color="lightgray")
view.center()
view

/WAVE/bio/ML/SAE_train/SAEProteinMPNN/evaluation/inputs/1a7w.pdb
True
73467


NGLWidget()